In [1]:
# user_logs에서 2017-01-31 이전 고객별 마지막 이용일 집계

import time
from pathlib import Path

import pandas as pd


# 경로 설정
CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path
        for path in [
            CURRENT_DIR,
            *CURRENT_DIR.parents,
        ]
        if (
            (path / "preprocessing").is_dir()
            and (path / "modeling").is_dir()
            and (path / ".gitignore").exists()
        )
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "SKN34-2nd-2Team-modify 경로를 찾을 수 없습니다."
    )

ORIGINAL_ROOT = PROJECT_ROOT

RAW_PATH = (
    ORIGINAL_ROOT
    / "data"
    / "raw"
    / "user_logs.csv"
)

PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

LABEL_PATH = (
    PROCESSED_DIR
    / "labels_pooled.csv"
)

OUTPUT_PATH = (
    PROCESSED_DIR
    / "features_user_logs_recency.csv"
)

CHECKPOINT_PATH = (
    PROCESSED_DIR
    / "user_logs_recency_checkpoint.pkl"
)

assert RAW_PATH.exists(), RAW_PATH
assert LABEL_PATH.exists(), LABEL_PATH

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# 설정
CUTOFF_INT = 20170131

CUTOFF_DATE = pd.Timestamp(
    "2017-01-31"
)

CHUNK_SIZE = 5_000_000

NO_LOG_VALUE = 999


# 예측 대상 고객만 집계
labels = pd.read_csv(
    LABEL_PATH,
    usecols=["msno"],
)

assert labels["msno"].duplicated().sum() == 0

target_users = set(labels["msno"])

print("원본 로그:", RAW_PATH)
print("대상 고객:", f"{len(target_users):,}")
print("Chunk 크기:", f"{CHUNK_SIZE:,}")
print("집계 시작\n")


# Chunk별 고객의 최대 로그 날짜 누적
start_time = time.time()

last_log_max = None
scanned_rows = 0
selected_rows = 0

reader = pd.read_csv(
    RAW_PATH,
    usecols=["msno", "date"],
    dtype={
        "msno": "string",
        "date": "int32",
    },
    chunksize=CHUNK_SIZE,
)

for chunk_number, chunk in enumerate(
    reader,
    start=1,
):
    scanned_rows += len(chunk)

    chunk = chunk[
        chunk["date"] <= CUTOFF_INT
    ]

    chunk = chunk[
        chunk["msno"].isin(target_users)
    ]

    selected_rows += len(chunk)

    chunk_max = (
        chunk.groupby(
            "msno",
            sort=False,
        )["date"]
        .max()
    )

    if last_log_max is None:
        last_log_max = chunk_max

    elif not chunk_max.empty:
        last_log_max = (
            pd.concat([
                last_log_max,
                chunk_max,
            ])
            .groupby(
                level=0,
                sort=False,
            )
            .max()
        )

    elapsed = time.time() - start_time

    print(
        f"chunk {chunk_number:02d} 완료"
        f" | 스캔 {scanned_rows:,}"
        f" | 대상 로그 {selected_rows:,}"
        f" | 대상 고객 {len(last_log_max):,}"
        f" | 경과 {elapsed / 60:.1f}분"
    )

    # 중간 결과 백업
    if chunk_number % 10 == 0:
        pd.to_pickle(
            {
                "completed_chunk": chunk_number,
                "scanned_rows": scanned_rows,
                "selected_rows": selected_rows,
                "last_log_max": last_log_max,
            },
            CHECKPOINT_PATH,
        )

        print(
            "  체크포인트 저장:",
            CHECKPOINT_PATH,
        )


# 사용자 단위 결과 생성
last_log_features = (
    last_log_max
    .rename("last_log_date_raw")
    .reset_index()
)

recency_features = labels.merge(
    last_log_features,
    on="msno",
    how="left",
    validate="one_to_one",
)

last_log_timestamp = pd.to_datetime(
    recency_features[
        "last_log_date_raw"
    ].astype("Int64").astype("string"),
    format="%Y%m%d",
    errors="coerce",
)

recency_features[
    "has_log_before_cutoff"
] = (
    last_log_timestamp.notna()
    .astype("int8")
)

recency_features[
    "days_since_last_log"
] = (
    CUTOFF_DATE
    - last_log_timestamp
).dt.days

recency_features[
    "days_since_last_log"
] = (
    recency_features[
        "days_since_last_log"
    ]
    .fillna(NO_LOG_VALUE)
    .astype("int32")
)

recency_features[
    "last_log_date"
] = (
    last_log_timestamp.dt.strftime(
        "%Y-%m-%d"
    )
)

recency_features = recency_features[[
    "msno",
    "last_log_date",
    "days_since_last_log",
    "has_log_before_cutoff",
]]


# 결과 검증
assert len(recency_features) == len(labels)

assert (
    recency_features["msno"]
    .duplicated()
    .sum()
    == 0
)

assert (
    recency_features[
        "days_since_last_log"
    ] < 0
).sum() == 0

assert (
    recency_features[
        "days_since_last_log"
    ].isna().sum()
    == 0
)


# 결과 저장
recency_features.to_csv(
    OUTPUT_PATH,
    index=False,
)

elapsed_minutes = (
    time.time() - start_time
) / 60

no_log_count = (
    recency_features[
        "has_log_before_cutoff"
    ] == 0
).sum()

print("\n===== 집계 완료 =====")
print("전체 스캔:", f"{scanned_rows:,}")
print("대상 로그:", f"{selected_rows:,}")
print("결과 shape:", recency_features.shape)
print("로그 없는 고객:", f"{no_log_count:,}")
print(
    "로그 없는 고객 비율:",
    f"{no_log_count / len(recency_features):.2%}",
)
print(
    "최대 실제 경과일:",
    recency_features.loc[
        recency_features[
            "has_log_before_cutoff"
        ] == 1,
        "days_since_last_log",
    ].max(),
)
print(
    "전체 소요시간:",
    f"{elapsed_minutes:.1f}분",
)
print("저장 완료:", OUTPUT_PATH)

원본 로그: C:\Users\playdata2\Desktop\SKN34-2nd-2team\data\raw\user_logs.csv
대상 고객: 992,931
Chunk 크기: 5,000,000
집계 시작

chunk 01 완료 | 스캔 5,000,000 | 대상 로그 2,973,906 | 대상 고객 484,882 | 경과 0.2분
chunk 02 완료 | 스캔 10,000,000 | 대상 로그 5,958,732 | 대상 고객 763,356 | 경과 0.4분
chunk 03 완료 | 스캔 15,000,000 | 대상 로그 8,934,459 | 대상 고객 765,552 | 경과 0.6분
chunk 04 완료 | 스캔 20,000,000 | 대상 로그 11,911,482 | 대상 고객 767,645 | 경과 0.8분
chunk 05 완료 | 스캔 25,000,000 | 대상 로그 14,890,446 | 대상 고객 769,827 | 경과 1.0분
chunk 06 완료 | 스캔 30,000,000 | 대상 로그 17,867,032 | 대상 고객 771,883 | 경과 1.2분
chunk 07 완료 | 스캔 35,000,000 | 대상 로그 20,845,096 | 대상 고객 773,963 | 경과 1.4분
chunk 08 완료 | 스캔 40,000,000 | 대상 로그 23,824,433 | 대상 고객 776,031 | 경과 1.8분
chunk 09 완료 | 스캔 45,000,000 | 대상 로그 26,800,840 | 대상 고객 778,078 | 경과 2.2분
chunk 10 완료 | 스캔 50,000,000 | 대상 로그 29,781,458 | 대상 고객 780,153 | 경과 2.5분
  체크포인트 저장: C:\Users\playdata2\Desktop\SKN34-2nd-2team\data\processed\user_logs_recency_checkpoint.pkl
chunk 11 완료 | 스캔 55,000,000 | 대상 로그 32,756,180 | 대상 고객 7

In [2]:
# 집계 결과 검증 후 enhanced_v1에 병합하여 v2 생성

from sklearn.metrics import roc_auc_score


RECENCY_PATH = (
    PROCESSED_DIR
    / "features_user_logs_recency.csv"
)

V1_PATH = (
    PROCESSED_DIR
    / "model_table_enhanced_v1.csv"
)

V2_PATH = (
    PROCESSED_DIR
    / "model_table_enhanced_v2.csv"
)


recency = pd.read_csv(
    RECENCY_PATH,
    low_memory=False,
)

enhanced_v1 = pd.read_csv(
    V1_PATH,
    low_memory=False,
)


# 집계 결과 기본 검증
assert recency.shape == (992931, 4)
assert recency["msno"].duplicated().sum() == 0
assert recency.isna().sum()[
    "days_since_last_log"
] == 0

assert (
    recency["days_since_last_log"] < 0
).sum() == 0

assert set(
    recency["has_log_before_cutoff"].unique()
).issubset({0, 1})


# v1과 msno 기준 병합
enhanced_v2 = enhanced_v1.merge(
    recency,
    on="msno",
    how="left",
    validate="one_to_one",
)

assert len(enhanced_v2) == len(enhanced_v1)
assert enhanced_v2[
    "days_since_last_log"
].isna().sum() == 0


# 기존 로그 존재 여부와 신규 집계 비교
log_flag_mismatch = (
    enhanced_v2["has_log_activity"]
    != enhanced_v2[
        "has_log_before_cutoff"
    ]
).sum()

print(
    "로그 존재 플래그 불일치:",
    f"{log_flag_mismatch:,}",
)


# 날짜 문자열과 중복 플래그는 모델에서 제외
# 모델에는 days_since_last_log만 추가
enhanced_v2 = enhanced_v2.drop(
    columns=[
        "last_log_date",
        "has_log_before_cutoff",
    ]
)

assert enhanced_v2.shape == (992931, 61)
assert enhanced_v2.isna().sum().sum() == 0
assert enhanced_v2["msno"].duplicated().sum() == 0
assert enhanced_v2["snapshot"].eq(
    "2017-01-31"
).all()


# Validation 단변량 AUC 확인
valid_recency = enhanced_v2[
    enhanced_v2["split"] == "valid"
]

recency_auc = roc_auc_score(
    valid_recency["is_churn"],
    valid_recency[
        "days_since_last_log"
    ],
)

directional_auc = max(
    recency_auc,
    1 - recency_auc,
)


# v2 저장
enhanced_v2.to_csv(
    V2_PATH,
    index=False,
)

actual_log_mask = (
    enhanced_v2["days_since_last_log"]
    != 999
)

print("\n===== Enhanced v2 검증 =====")
print("shape:", enhanced_v2.shape)
print(
    "로그 없는 고객:",
    f"{(~actual_log_mask).sum():,}",
)
print(
    "실제 경과일 최소:",
    enhanced_v2.loc[
        actual_log_mask,
        "days_since_last_log",
    ].min(),
)
print(
    "실제 경과일 최대:",
    enhanced_v2.loc[
        actual_log_mask,
        "days_since_last_log",
    ].max(),
)
print(
    "days_since_last_log AUC:",
    f"{recency_auc:.6f}",
)
print(
    "Directional AUC:",
    f"{directional_auc:.6f}",
)
print("저장 완료:", V2_PATH)

로그 존재 플래그 불일치: 0

===== Enhanced v2 검증 =====
shape: (992931, 61)
로그 없는 고객: 127,196
실제 경과일 최소: 0
실제 경과일 최대: 761
days_since_last_log AUC: 0.513228
Directional AUC: 0.513228
저장 완료: C:\Users\playdata2\Desktop\SKN34-2nd-2team\data\processed\model_table_enhanced_v2.csv
